# 01 — Build revision-pair dataset

Constructs paired (first, second) summary observations per (user_id, page_slug), for both STAIRS and `random_reread` conditions. Follows the extraction pattern in `itell-data/2024.05-CTTC/src/Checkout.ipynb`.

In [ ]:
import os, sys, ast, json
import pandas as pd

DATA = os.path.expanduser('~/active-projects/itell-data/2024.05-CTTC/data/supabase_filtered')
OUT = os.path.join(os.path.dirname(os.getcwd()), 'data') if os.path.basename(os.getcwd()) == 'src' else 'data'
os.makedirs(OUT, exist_ok=True)
print('DATA =', DATA)
print('OUT  =', OUT)

In [ ]:
summaries = pd.read_csv(os.path.join(DATA, 'summaries.csv.gz'))
users = pd.read_csv(os.path.join(DATA, 'users.csv.gz'))[['id','condition','prolific_pid']].rename(columns={'id':'user_id'})
summaries['created_at'] = pd.to_datetime(summaries['created_at'])
print('summaries', summaries.shape)
print('users', users.shape)
print(summaries['condition'].value_counts().to_dict())

## Extract first and second submissions per (user, page)

In [ ]:
# First = earliest per (user, page); Second = earliest among the remainder.
first_idx = summaries.groupby(['user_id','page_slug'])['created_at'].idxmin()
first = summaries.loc[first_idx].copy()

rest = summaries.drop(index=first_idx)
second_idx = rest.groupby(['user_id','page_slug'])['created_at'].idxmin()
second = rest.loc[second_idx].copy()

print('first', first.shape, 'second', second.shape)

In [ ]:
keep = ['id','text','user_id','page_slug','condition','isPassed',
        'containment_score','similarity_score','language_score','content_score','created_at']
f = first[keep].rename(columns={c: c+'_t1' for c in keep if c not in ('user_id','page_slug','condition')})
s = second[keep].rename(columns={c: c+'_t2' for c in keep if c not in ('user_id','page_slug','condition')})

# Inner-join so we only keep user/page with BOTH t1 and t2.
pairs = f.merge(s.drop(columns=['condition']), on=['user_id','page_slug'], how='inner')
print('pairs', pairs.shape)
print(pairs['condition'].value_counts().to_dict())

In [ ]:
# Plan expects 157 STAIRS pairs and 110 random_reread pairs.
expected = {'stairs': 157, 'random_reread': 110}
got = pairs['condition'].value_counts().to_dict()
assert got == expected, f'Pair counts {got} != expected {expected}'
print('OK — pair counts match plan')

## Link STAIRS feedback text via timestamp windowing

`chat_messages.data` is a stringified list of turns, each with `isStairs`, `isUser`, and a unix-ms `timestamp`. We pull the non-user STAIRS turns whose timestamp falls strictly between `created_at_t1` and `created_at_t2` for each pair.

In [ ]:
chat = pd.read_csv(os.path.join(DATA, 'chat_messages.csv.gz'))
chat['created_at'] = pd.to_datetime(chat['created_at'])
print('chat rows', len(chat))

records = []
for row in chat.itertuples(index=False):
    try:
        turns = ast.literal_eval(row.data)
    except Exception:
        continue
    for turn in turns:
        if not isinstance(turn, dict):
            continue
        records.append({
            'user_id': row.user_id,
            'page_slug_full': row.page_slug,
            'text': turn.get('text',''),
            'isUser': turn.get('isUser', False),
            'isStairs': turn.get('isStairs', False),
            'timestamp_ms': turn.get('timestamp'),
        })
turns_df = pd.DataFrame(records)
turns_df['turn_time'] = pd.to_datetime(turns_df['timestamp_ms'], unit='ms', utc=True)
# Match on the full slug string, which is what summaries.page_slug contains.
turns_df['page_slug'] = turns_df['page_slug_full']
print('turn rows', len(turns_df))
print(turns_df[['isUser','isStairs']].value_counts().to_dict())

In [ ]:
# For each STAIRS pair, collect the assistant (non-user) STAIRS turns that
# fall between t1 and t2.
def feedback_for(row):
    if row.condition != 'stairs':
        return ''
    sub = turns_df[(turns_df['user_id']==row.user_id) &
                   (turns_df['page_slug']==row.page_slug) &
                   (~turns_df['isUser']) & (turns_df['isStairs']) &
                   (turns_df['turn_time']>=row.created_at_t1) &
                   (turns_df['turn_time']<=row.created_at_t2)]
    return '\n---\n'.join(sub['text'].astype(str).tolist())

pairs['stairs_feedback'] = pairs.apply(feedback_for, axis=1)
has_fb = (pairs['condition']=='stairs') & (pairs['stairs_feedback'].str.len()>0)
print('STAIRS pairs with linked feedback:', int(has_fb.sum()), '/', int((pairs.condition=='stairs').sum()))

In [ ]:
out_path = os.path.join(OUT, 'revision_pairs.parquet')
pairs.to_parquet(out_path, index=False)
print('wrote', out_path, pairs.shape)